In [1]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [13]:
from textblob import TextBlob
import pandas as pd

df = pd.read_csv('hotel_reviews.csv')
print("Shape:", df.shape)
df.head()

Shape: (7001, 7)


,Index,Name,Area,Review_Date,Rating_attribute,Rating(Out of 10),Review_Text
0,0,Hotel The Pearl,"Paharganj, New Delhi",Jul-23,Best budget friendly hotel,9.0,Hotel the pearl is perfect place to stay in De...
1,1,Hotel The Pearl,"Paharganj, New Delhi",Aug-23,Amazing place,9.0,Location of the hotel is perfect. The hotel is...
2,2,Hotel The Pearl,"Paharganj, New Delhi",Aug-23,Overall good stay. Economic.,9.0,"Location, Indian food."
3,3,Hotel The Pearl,"Paharganj, New Delhi",Aug-23,Lovely,9.0,The location and the hotel itself is great. Ne...
4,4,Hotel The Pearl,"Paharganj, New Delhi",Aug-23,Great hotel Great staff and great staying,9.0,Friendly and smiling staffs.. The reception st...


In [8]:
df.tail()

,Index,Name,Area,Review_Date,Rating_attribute,Rating(Out of 10),Review_Text
6996,6996,FabHotel F9 NSP,"North Delhi, New Delhi",Aug-23,I'd like to thank Manager.,10.0,"The room was good, comfortable and aesthetic \..."
6997,6997,FabHotel F9 NSP,"North Delhi, New Delhi",Jul-23,Superb,9.0,good hotel
6998,6998,FabHotel F9 NSP,"North Delhi, New Delhi",Jul-23,fabulous,10.0,good experience for me about hotel \nvery good...
6999,6999,FabHotel F9 NSP,"North Delhi, New Delhi",Jun-23,well done,10.0,well done
7000,7000,FabHotel F9 NSP,"North Delhi, New Delhi",Jul-23,Bad,2.0,Nothing


In [15]:
import pandas as pd
import re
import nltk

nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words
             if word not in stop_words]
    return " ".join(words)

df['Cleaned_Review'] = df['Review_Text'].apply(preprocess)

print(df[['Review_Text', 'Cleaned_Review']].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


                                         Review_Text  \
0  Hotel the pearl is perfect place to stay in De...   
1  Location of the hotel is perfect. The hotel is...   
2                             Location, Indian food.   
3  The location and the hotel itself is great. Ne...   
4  Friendly and smiling staffs.. The reception st...   

                                      Cleaned_Review  
0  hotel pearl perfect place stay delhi paharganj...  
1  location hotel perfect hotel peaceful nice sta...  
2                               location indian food  
3  location hotel great next time stay nice room ...  
4  friendly smiling staff reception staff excelle...  


In [16]:
!pip install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(review):
    score = analyzer.polarity_scores(review)['compound']

    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['Sentiment'] = df['Cleaned_Review'].apply(get_sentiment)

print(df['Sentiment'].value_counts())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.6 MB/s eta 0:00:00
Sentiment
Positive    4593
Neutral     1915
Negative     493
Name: count, dtype: int64


In [18]:
categories = {
    'Room Quality': ['room', 'bed', 'bathroom', 'ac'],
    'Staff Behavior': ['staff', 'manager', 'receptionist', 'service'],
    'Cleanliness': ['clean', 'dirty', 'hygiene'],
    'Food Service': ['food', 'breakfast', 'restaurant', 'meal'],
    'Amenities': ['wifi', 'pool', 'gym', 'parking']
}

def identify_issues(review):
    issues = []

    for category, keywords in categories.items():
        for word in keywords:
            if word in review:
                issues.append(category)
                break

    return ', '.join(issues)

df['Issues'] = df['Cleaned_Review'].apply(identify_issues)

print(df[['Review_Text', 'Issues']].head())

                                         Review_Text  \
0  Hotel the pearl is perfect place to stay in De...   
1  Location of the hotel is perfect. The hotel is...   
2                             Location, Indian food.   
3  The location and the hotel itself is great. Ne...   
4  Friendly and smiling staffs.. The reception st...   

                                      Issues  
0  Room Quality, Staff Behavior, Cleanliness  
1               Room Quality, Staff Behavior  
2                               Food Service  
3               Room Quality, Staff Behavior  
4               Room Quality, Staff Behavior  


In [19]:
sentiment_counts = df['Sentiment'].value_counts()

print("\nCustomer Satisfaction Report")
print("----------------------------")
print(sentiment_counts)


Customer Satisfaction Report
----------------------------
Sentiment
Positive    4593
Neutral     1915
Negative     493
Name: count, dtype: int64
